# Member 1 — Data Audit and Dev Splits
**Project:** Revisiting MARBERT for Arabic Sentiment through Reproducibility and Linguistic Robustness (AI504, Assignment 2)

**What this notebook does (run the cells from top to bottom):**
1. Downloads the three datasets (ArSarcasm, AJGT, ASTD) and records their versions.
2. Maps every original label to one standard form: `negative`, `neutral`, `positive`.
3. Counts tweets per split, class, dialect, sarcasm and source.
4. Finds duplicates inside each dataset and the same tweet across datasets (cross-source overlap).
5. Creates the train/dev/test split files for Members 2 and 3 (stratified 10% dev, fixed seed, test never changed).
6. Saves all audit tables, figures and a short report, and zips everything.

**How to run in Google Colab:** `Runtime → Run all`. No GPU is needed. It takes about 2–5 minutes.

In [ ]:
# ---------- 1. Settings ----------
import os, re, io, glob, json, hashlib, zipfile, subprocess, unicodedata, datetime, platform
import numpy as np
import pandas as pd

SEED = 42            # fixed random seed: anyone who runs this gets exactly the same splits
DEV_FRAC = 0.10      # 10% of each training set becomes the dev (practice-exam) set
TEST_FRAC = 0.20     # used only for datasets with no official test split (AJGT, ASTD), as in the paper's 80/20 setup
LABELS = ["negative", "neutral", "positive"]

RAW = "raw"                 # downloaded original files
OUT = "member1_outputs"     # everything this notebook produces
SPLITS = os.path.join(OUT, "splits")
TABLES = os.path.join(OUT, "audit_tables")
FIGS = os.path.join(OUT, "figures")
EXTRA = "extra_sources"     # OPTIONAL: put ArSAS / SemEval-2017 / ArSenTD-LEV CSVs here (see Section 7)
for d in [RAW, OUT, SPLITS, TABLES, FIGS, EXTRA]:
    os.makedirs(d, exist_ok=True)

print("Python", platform.python_version(), "| pandas", pd.__version__)
print("Run date:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))

In [ ]:
# ---------- 2. Download the datasets (and record their versions) ----------
SOURCES = {
    "ArSarcasm": {"git": "https://github.com/iabufarha/ArSarcasm.git", "hf": "iabufarha/ar_sarcasm"},
    "ASTD":      {"git": "https://github.com/mahmoudnabil/ASTD.git",   "hf": "arbml/ASTD"},
    "AJGT":      {"url": "https://raw.githubusercontent.com/komari6/Arabic-twitter-corpus-AJGT/master/AJGT.xlsx",
                  "hf": "komari6/ajgt_twitter_ar"},
}
versions = []   # one row per dataset: where it came from and which exact version

def run(cmd):
    """Run a shell command and return (ok, output)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

def hf_parquet(repo_id, dest):
    """Fallback: download the Hugging Face auto-converted Parquet files of a dataset."""
    from huggingface_hub import snapshot_download
    p = snapshot_download(repo_id=repo_id, repo_type="dataset", revision="refs/convert/parquet", local_dir=dest)
    return sorted(glob.glob(os.path.join(p, "**", "*.parquet"), recursive=True))

for name in ["ArSarcasm", "ASTD"]:
    dest = os.path.join(RAW, name)
    if not os.path.isdir(dest) or not os.listdir(dest):
        ok, msg = run(f"git clone --depth 1 {SOURCES[name]['git']} {dest}")
        print(name, "git clone:", "OK" if ok else "FAILED -> " + msg[-200:])
    ok, commit = run(f"git -C {dest} rev-parse HEAD")
    versions.append({"dataset": name, "source": SOURCES[name]["git"],
                     "version": ("git commit " + commit[:12]) if ok else "local copy (no git info)"})

ajgt_path = os.path.join(RAW, "AJGT.xlsx")
if not os.path.exists(ajgt_path):
    ok, msg = run(f"wget -q -O {ajgt_path} {SOURCES['AJGT']['url']}")
    print("AJGT download:", "OK" if ok else "FAILED -> " + msg[-200:])
versions.append({"dataset": "AJGT", "source": SOURCES["AJGT"]["url"],
                 "version": ("file sha256 " + sha256(ajgt_path)) if os.path.exists(ajgt_path) and os.path.getsize(ajgt_path) > 0 else "missing"})

for v in versions:
    v["downloaded"] = datetime.date.today().isoformat()
pd.DataFrame(versions)

In [ ]:
# ---------- 3. Load each dataset and map its labels ----------
label_maps = []   # table: dataset, original label, count, mapped label, kept?

def record_mapping(dataset, raw_series, mapping):
    for raw_label, n in raw_series.value_counts(dropna=False).sort_index().items():
        mapped = mapping.get(raw_label)
        label_maps.append({"dataset": dataset, "original_label": repr(raw_label), "count": int(n),
                           "mapped_to": mapped if mapped else "(removed)", "kept": bool(mapped)})

# --- ArSarcasm: official train/test CSV files; columns dialect, sarcasm, sentiment, original_sentiment, tweet, source
def load_arsarcasm():
    files = sorted(glob.glob(os.path.join(RAW, "ArSarcasm", "**", "*.csv"), recursive=True))
    train_f = [f for f in files if re.search(r"train", os.path.basename(f), re.I)]
    test_f = [f for f in files if re.search(r"test", os.path.basename(f), re.I)]
    parts = []
    if train_f and test_f:
        for split, f in [("train", train_f[0]), ("test", test_f[0])]:
            d = pd.read_csv(f, dtype=str, keep_default_na=False)
            d["split"], d["file"] = split, os.path.relpath(f, RAW)
            parts.append(d)
    else:  # fallback to Hugging Face parquet
        for f in hf_parquet(SOURCES["ArSarcasm"]["hf"], os.path.join(RAW, "ArSarcasm_hf")):
            d = pd.read_parquet(f).astype(str)
            d["split"] = "train" if "train" in f else "test"
            d["file"] = os.path.relpath(f, RAW)
            parts.append(d)
    df = pd.concat(parts, ignore_index=True)
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={"tweet": "text"})
    df["sarcasm"] = df["sarcasm"].astype(str).str.strip().str.lower().isin(["true", "1", "sarcastic"])
    df["dialect"] = df["dialect"].astype(str).str.strip().str.lower()
    df["source"] = df["source"].astype(str).str.strip().str.lower()
    raw = df["sentiment"].astype(str).str.strip()
    mapping = {"negative": "negative", "neutral": "neutral", "positive": "positive",
               "NEG": "negative", "NEU": "neutral", "POS": "positive", "0": "negative", "1": "neutral", "2": "positive"}
    record_mapping("ArSarcasm", raw, mapping)
    df["label"] = raw.map(mapping)
    df["orig_row"] = df.groupby("split").cumcount()
    df["id"] = "arsarcasm_" + df["split"] + "_" + df["orig_row"].astype(str).str.zfill(5)
    return df

# --- AJGT: one Excel file; columns Feed (tweet) and Sentiment (Positive / Negative); no official split
def load_ajgt():
    if os.path.exists(ajgt_path) and os.path.getsize(ajgt_path) > 0:
        d = pd.read_excel(ajgt_path, dtype=str)
        d["file"] = "AJGT.xlsx"
    else:
        d = pd.concat([pd.read_parquet(f) for f in hf_parquet(SOURCES["AJGT"]["hf"], os.path.join(RAW, "AJGT_hf"))])
        d = d.rename(columns={"text": "Feed", "label": "Sentiment"}).astype(str)
        d["Sentiment"] = d["Sentiment"].replace({"0": "Negative", "1": "Positive"})
        d["file"] = "huggingface parquet"
    d.columns = [c.strip() for c in d.columns]
    raw = d["Sentiment"].astype(str).str.strip()
    mapping = {"Positive": "positive", "Negative": "negative", "positive": "positive", "negative": "negative"}
    record_mapping("AJGT", d["Sentiment"].astype(str), {k: mapping.get(k.strip()) for k in d["Sentiment"].astype(str).unique()})
    out = pd.DataFrame({"text": d["Feed"].astype(str), "label": raw.map(mapping), "file": d["file"]})
    out["orig_row"] = range(len(out))
    out["id"] = "ajgt_" + out["orig_row"].astype(str).str.zfill(5)
    out["split"] = "all"
    return out

# --- ASTD: Tweets.txt, one tweet per line: "<tweet><TAB><label>"; labels OBJ, POS, NEG, NEUTRAL; no 3-class split released
def load_astd():
    files = glob.glob(os.path.join(RAW, "ASTD", "**", "Tweets.txt"), recursive=True)
    rows = []
    if files:
        with open(files[0], encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.rstrip("\n\r")
                if not line.strip():
                    continue
                if "\t" in line:
                    text, lab = line.rsplit("\t", 1)
                else:
                    text, lab = line, ""
                rows.append({"orig_row": i, "text": text, "raw_label": lab.strip(), "file": os.path.relpath(files[0], RAW)})
        d = pd.DataFrame(rows)
    else:  # fallback: Hugging Face arbml/ASTD (labels 0 Neutral, 1 Objective, 2 Positive, 3 Negative)
        d = pd.concat([pd.read_parquet(f) for f in hf_parquet(SOURCES["ASTD"]["hf"], os.path.join(RAW, "ASTD_hf"))])
        d = d.rename(columns={"tweet": "text"})
        d["raw_label"] = d["label"].astype(str).replace({"0": "NEUTRAL", "1": "OBJ", "2": "POS", "3": "NEG"})
        d["orig_row"] = range(len(d)); d["file"] = "huggingface parquet"
    # The paper follows CAMeL Tools: objective tweets are discarded; positive / negative / neutral are kept.
    mapping = {"POS": "positive", "NEG": "negative", "NEUTRAL": "neutral", "OBJ": None}
    record_mapping("ASTD", d["raw_label"], mapping)
    d["label"] = d["raw_label"].map(mapping)
    d["id"] = "astd_" + d["orig_row"].astype(str).str.zfill(5)
    d["split"] = "all"
    return d

ars = load_arsarcasm()
ajgt = load_ajgt()
astd_all = load_astd()
print("Loaded rows -> ArSarcasm:", len(ars), "| AJGT:", len(ajgt), "| ASTD (all classes):", len(astd_all))

label_map_df = pd.DataFrame(label_maps)
label_map_df

In [ ]:
# ---------- 4. Basic quality checks: empty texts, unmapped labels, text length ----------
def text_checks(name, d):
    t = d["text"].fillna("").astype(str)
    return {"dataset": name, "rows": len(d),
            "empty_text": int((t.str.strip() == "").sum()),
            "removed_or_unmapped_label": int(d["label"].isna().sum()),
            "mean_words": round(t.str.split().str.len().mean(), 1),
            "max_words": int(t.str.split().str.len().max()) if len(t) else 0}

quality_df = pd.DataFrame([text_checks("ArSarcasm", ars), text_checks("AJGT", ajgt),
                           text_checks("ASTD (all classes)", astd_all)])
astd = astd_all[astd_all["label"].notna()].copy()   # ASTD after removing objective tweets
assert len(astd) > 0, "ASTD has no positive/negative/neutral tweets: check the label mapping table above"
print("ASTD after removing objective tweets:", len(astd))
quality_df

In [ ]:
# ---------- 5. Text normalisation used ONLY to detect duplicates (the saved text is never changed) ----------
DIAC = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")   # Arabic diacritics
def norm(t):
    t = unicodedata.normalize("NFKC", str(t))
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)      # links
    t = re.sub(r"@\w+", " ", t)                        # user mentions
    t = re.sub(r"\bRT\b", " ", t)                      # retweet marker
    t = DIAC.sub("", t).replace("ـ", "")          # diacritics and tatweel
    t = re.sub("[إأآٱ]", "ا", t).replace("ى", "ي").replace("ة", "ه")
    t = re.sub(r"[^\w\s]", " ", t)                     # punctuation and emoji
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)               # long letter repeats (جمييييل -> جمييل)
    return re.sub(r"\s+", " ", t).strip().lower()

for d in (ars, ajgt, astd):
    d["norm"] = d["text"].map(norm)
    d["text_exact"] = d["text"].astype(str).str.strip()
print("Example:", repr(ars["text"].iloc[0][:80]), "->", repr(ars["norm"].iloc[0][:80]))

In [ ]:
# ---------- 6. Duplicates inside each dataset ----------
def dup_summary(name, d, key):
    g = d[d[key] != ""].groupby(key)
    sizes = g.size()
    groups = sizes[sizes > 1]
    conflicting = g["label"].nunique()
    conflicting = conflicting[conflicting > 1]
    return {"dataset": name, "match_type": "exact text" if key == "text_exact" else "normalised text",
            "rows": len(d), "duplicate_groups": int(len(groups)),
            "extra_copies": int((groups - 1).sum()),
            "groups_with_conflicting_labels": int(len(conflicting))}

dups = []
for name, d in [("ArSarcasm", ars), ("AJGT", ajgt), ("ASTD (3-class)", astd)]:
    for key in ["text_exact", "norm"]:
        dups.append(dup_summary(name, d, key))
dup_df = pd.DataFrame(dups)

# Leakage inside ArSarcasm's official split: the same tweet in both train and test
tr_norm = set(ars.loc[ars.split == "train", "norm"]) - {""}
ars_leak = ars[(ars.split == "test") & ars["norm"].isin(tr_norm)]
print("ArSarcasm test tweets that also appear in ArSarcasm train (normalised):", len(ars_leak))

# Example conflicting-label duplicates (same tweet, different labels), for the report
def conflict_examples(d, n=5):
    g = d[d.norm != ""].groupby("norm")["label"].nunique()
    keys = g[g > 1].index[:n]
    return d[d.norm.isin(keys)][["id", "text", "label"]].sort_values("text")
conflicts_ars = conflict_examples(ars)
dup_df

In [ ]:
# ---------- 7. OPTIONAL: other sources of the merged CAMeL-Sent training corpus ----------
# In the paper, the ASTD model is trained on a merged corpus (train + dev of ArSAS, ASTD, SemEval-2017, ArSenTD-LEV,
# about 24.8K tweets) and tested on the held-out ASTD test set. ArSAS, SemEval-2017 Task 4 and ArSenTD-LEV are not
# downloadable from GitHub and some need a request form. If the team obtains them, save each one in the folder
# "extra_sources/" as a CSV with the columns: text, label (negative / neutral / positive), split (train / dev / test).
# The file name becomes the source name, e.g. extra_sources/arsas.csv. If the folder is empty, this cell does nothing.
extra = []
for f in sorted(glob.glob(os.path.join(EXTRA, "*.csv"))):
    e = pd.read_csv(f, dtype=str, keep_default_na=False)
    e["source_name"] = os.path.splitext(os.path.basename(f))[0]
    e["label"] = e["label"].str.strip().str.lower()
    e = e[e["label"].isin(LABELS)].copy()
    e["norm"] = e["text"].map(norm)
    e["id"] = e["source_name"] + "_" + pd.Series(range(len(e)), index=e.index).astype(str).str.zfill(5)
    extra.append(e)
extra_df = pd.concat(extra, ignore_index=True) if extra else pd.DataFrame(columns=["text", "label", "split", "source_name", "norm", "id"])
print("Extra CAMeL-Sent sources found:", sorted(extra_df["source_name"].unique()) if len(extra_df) else "none")

In [ ]:
# ---------- 8. Create the splits (test sets are never changed after this cell) ----------
from sklearn.model_selection import StratifiedGroupKFold

def grouped_holdout(d, frac, seed=SEED):
    """Stratified split that keeps all copies of the same (normalised) tweet on the same side.
    Returns (keep_index, holdout_index); the holdout is about `frac` of the rows."""
    k = int(round(1 / frac))
    groups = d["norm"].where(d["norm"] != "", d["id"])        # empty texts are their own group
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    keep, hold = next(sgkf.split(d, d["label"], groups))
    return d.index[keep], d.index[hold]

# ArSarcasm: official train/test kept; dev = 10% of train
ars_train_full = ars[ars.split == "train"]
ars_test = ars[ars.split == "test"]
k_idx, h_idx = grouped_holdout(ars_train_full, DEV_FRAC)
ars_train, ars_dev = ars_train_full.loc[k_idx], ars_train_full.loc[h_idx]

# AJGT: no official split -> 80/20 train/test (as in the paper's setup), then dev = 10% of train
k_idx, h_idx = grouped_holdout(ajgt, TEST_FRAC)
ajgt_train_full, ajgt_test = ajgt.loc[k_idx], ajgt.loc[h_idx]
k_idx, h_idx = grouped_holdout(ajgt_train_full, DEV_FRAC)
ajgt_train, ajgt_dev = ajgt_train_full.loc[k_idx], ajgt_train_full.loc[h_idx]

# ASTD (3-class): no released split -> 80/20 random split as described by CAMeL Tools, then dev = 10% of train
k_idx, h_idx = grouped_holdout(astd, TEST_FRAC)
astd_train_full, astd_test = astd.loc[k_idx], astd.loc[h_idx]
k_idx, h_idx = grouped_holdout(astd_train_full, DEV_FRAC)
astd_train, astd_dev = astd_train_full.loc[k_idx], astd_train_full.loc[h_idx]

# CAMeL-Sent merged training (only if extra sources were provided): ASTD train + other sources' train/dev
if len(extra_df):
    ex_tr = extra_df[extra_df["split"].str.lower().isin(["train", "dev"])].copy()
    ex_tr = ex_tr[~ex_tr["norm"].isin(set(astd_test["norm"]))]   # never let an ASTD test tweet into training
    camel_train_full = pd.concat([astd_train_full.assign(source_name="astd"), ex_tr], ignore_index=True)
    k_idx, h_idx = grouped_holdout(camel_train_full, DEV_FRAC)
    camel_train, camel_dev = camel_train_full.loc[k_idx], camel_train_full.loc[h_idx]

def check_disjoint(name, *parts):
    sets = [set(p["id"]) for p in parts]
    norms = [set(p["norm"]) - {""} for p in parts]
    assert all(not (a & b) for i, a in enumerate(sets) for b in sets[i + 1:]), name + ": an id is in two splits!"
    return {"dataset": name, "train/dev shared tweets": len(norms[0] & norms[1]),
            "train/test shared tweets": len(norms[0] & norms[2]), "dev/test shared tweets": len(norms[1] & norms[2])}

disjoint_df = pd.DataFrame([check_disjoint("ArSarcasm", ars_train, ars_dev, ars_test),
                            check_disjoint("AJGT", ajgt_train, ajgt_dev, ajgt_test),
                            check_disjoint("ASTD (3-class)", astd_train, astd_dev, astd_test)])
print("Split check (should be 0 except ArSarcasm train/test, which comes from the official files):")
disjoint_df

In [ ]:
# ---------- 9. Count everything ----------
def class_counts(name, parts):
    rows = []
    for split, p in parts.items():
        vc = p["label"].value_counts()
        row = {"dataset": name, "split": split, "total": len(p)}
        for l in LABELS:
            row[l] = int(vc.get(l, 0))
        rows.append(row)
    return rows

counts = []
counts += class_counts("ArSarcasm", {"train_full": ars_train_full, "train": ars_train, "dev": ars_dev, "test": ars_test})
counts += class_counts("AJGT", {"all": ajgt, "train_full": ajgt_train_full, "train": ajgt_train, "dev": ajgt_dev, "test": ajgt_test})
counts += class_counts("ASTD (3-class)", {"all": astd, "train_full": astd_train_full, "train": astd_train, "dev": astd_dev, "test": astd_test})
if len(extra_df):
    counts += class_counts("CAMeL-Sent (merged)", {"train_full": camel_train_full, "train": camel_train, "dev": camel_dev, "test (ASTD)": astd_test})
counts_df = pd.DataFrame(counts)

# Paper's sizes (Appendix Table A.1) vs ours
paper_vs_ours = pd.DataFrame([
    {"dataset": "ArSarcasm-Sent", "paper_train": "8.4K", "paper_test": "2.1K", "our_train_full": len(ars_train_full), "our_test": len(ars_test)},
    {"dataset": "ASTD (3-class)", "paper_train": "24.8K (merged CAMeL-Sent)", "paper_test": "664",
     "our_train_full": len(camel_train_full) if len(extra_df) else f"{len(astd_train_full)} (ASTD only)", "our_test": len(astd_test)},
    {"dataset": "AJGT", "paper_train": "1.4K", "paper_test": "361", "our_train_full": len(ajgt_train_full), "our_test": len(ajgt_test)},
])

# ArSarcasm subgroups: dialect x sentiment and sarcasm x sentiment, per split (with counts)
dialect_df = pd.crosstab([ars["split"], ars["dialect"]], ars["label"], margins=True, margins_name="total")
sarcasm_df = pd.crosstab([ars["split"], ars["sarcasm"].map({True: "sarcastic", False: "non-sarcastic"})], ars["label"], margins=True, margins_name="total")
source_df = pd.crosstab(ars["source"], ars["split"], margins=True, margins_name="total")
try:
    display(counts_df)          # nice table in Colab
except NameError:
    print(counts_df)
paper_vs_ours

In [ ]:
# ---------- 10. Cross-source overlap: the same tweet in two datasets ----------
# ArSarcasm tweets were originally taken from ASTD and SemEval-2017, and the ASTD (CAMeL-Sent) training data also
# contains ASTD and SemEval. So a tweet can be in the TRAINING data of one experiment and the TEST data of another.
def overlap(name_a, a, name_b, b):
    sa, sb = set(a["norm"]) - {""}, set(b["norm"]) - {""}
    shared = sa & sb
    return {"set_A": name_a, "set_B": name_b, "size_A": len(sa), "size_B": len(sb),
            "shared_tweets": len(shared), "share_of_A_%": round(100 * len(shared) / max(len(sa), 1), 2)}

pairs = [("ArSarcasm test", ars_test, "ASTD train_full", astd_train_full),
         ("ArSarcasm test", ars_test, "ASTD test", astd_test),
         ("ASTD test", astd_test, "ArSarcasm train_full", ars_train_full),
         ("ArSarcasm (all)", ars, "ASTD (all 3-class)", astd),
         ("ArSarcasm (all)", ars, "AJGT (all)", ajgt),
         ("AJGT (all)", ajgt, "ASTD (all 3-class)", astd)]
if len(extra_df):
    for s in sorted(extra_df["source_name"].unique()):
        e = extra_df[extra_df["source_name"] == s]
        pairs += [("ArSarcasm test", ars_test, f"{s} (all)", e), ("ASTD test", astd_test, f"{s} (all)", e)]
    pairs += [("ArSarcasm test", ars_test, "CAMeL-Sent train_full", camel_train_full)]
overlap_df = pd.DataFrame([overlap(*p) for p in pairs])

# Overlapping ArSarcasm test tweets whose label differs from the ASTD label (same tweet, different annotation)
m = ars_test.merge(astd[["norm", "label"]].drop_duplicates("norm"), on="norm", suffixes=("_arsarcasm", "_astd"))
label_disagree = m[m["label_arsarcasm"] != m["label_astd"]]
print("ArSarcasm test tweets also in ASTD:", len(m), "| of these, labelled differently:", len(label_disagree))
overlap_df

In [ ]:
# ---------- 11. Save the split files for Members 2 and 3 ----------
KEEP = ["id", "text", "label"]
ARS_EXTRA = ["dialect", "sarcasm", "source", "original_sentiment"]

def save(d, name, extra_cols=()):
    cols = KEEP + [c for c in extra_cols if c in d.columns]
    d[cols].sort_values("id").to_csv(os.path.join(SPLITS, name), index=False, encoding="utf-8")
    return {"file": name, "rows": len(d)}

saved = []
for split, d in [("train_full", ars_train_full), ("train", ars_train), ("dev", ars_dev), ("test", ars_test)]:
    saved.append(save(d, f"arsarcasm_{split}.csv", ARS_EXTRA))
for split, d in [("train_full", ajgt_train_full), ("train", ajgt_train), ("dev", ajgt_dev), ("test", ajgt_test)]:
    saved.append(save(d, f"ajgt_{split}.csv"))
for split, d in [("train_full", astd_train_full), ("train", astd_train), ("dev", astd_dev), ("test", astd_test)]:
    saved.append(save(d, f"astd_{split}.csv"))
if len(extra_df):
    for split, d in [("train_full", camel_train_full), ("train", camel_train), ("dev", camel_dev)]:
        saved.append(save(d, f"camelsent_{split}.csv", ["source_name"]))
saved_df = pd.DataFrame(saved)
print("train_full = train + dev (for the paper's historical protocol); train/dev/test = for the independent evaluation.")
saved_df

In [ ]:
# ---------- 12. Figures: class distribution per dataset and split ----------
import matplotlib
import matplotlib.pyplot as plt
COLORS = {"negative": "#2F6FD0", "neutral": "#9AA0A6", "positive": "#E8673C"}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for ax, (name, parts) in zip(axes, [("ArSarcasm", {"train": ars_train, "dev": ars_dev, "test": ars_test}),
                                    ("AJGT", {"train": ajgt_train, "dev": ajgt_dev, "test": ajgt_test}),
                                    ("ASTD (3-class)", {"train": astd_train, "dev": astd_dev, "test": astd_test})]):
    present = [l for l in LABELS if any((p["label"] == l).any() for p in parts.values())]   # AJGT has no neutral
    x = np.arange(len(parts)); w = 0.8 / len(present)
    for i, l in enumerate(present):
        vals = [int((p["label"] == l).sum()) for p in parts.values()]
        bars = ax.bar(x + (i - (len(present) - 1) / 2) * w, vals, w, label=l, color=COLORS[l], edgecolor="white", linewidth=1)
        ax.bar_label(bars, fontsize=7, padding=1)
    ax.set_xticks(x); ax.set_xticklabels(parts.keys()); ax.set_title(name)
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Number of tweets")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIGS, "class_distribution.png"), dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
dt = pd.crosstab(ars_test["dialect"], ars_test["label"]).reindex(columns=LABELS, fill_value=0)
bottom = np.zeros(len(dt))
for l in LABELS:
    ax.barh(dt.index, dt[l], left=bottom, color=COLORS[l], label=l, edgecolor="white", linewidth=1.5)
    bottom += dt[l].values
for i, tot in enumerate(bottom):
    ax.text(tot + bottom.max() * 0.01, i, f"n={int(tot)}", va="center", fontsize=8)
ax.set_xlim(0, bottom.max() * 1.12)
ax.set_title("ArSarcasm test set: tweets per dialect and sentiment"); ax.set_xlabel("Number of tweets")
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.22))
plt.tight_layout(); plt.savefig(os.path.join(FIGS, "arsarcasm_test_dialects.png"), dpi=200); plt.show()

In [ ]:
# ---------- 13. Save all audit tables (one Excel file + CSVs) and a short report ----------
tables = {"1_versions": pd.DataFrame(versions), "2_label_mapping": label_map_df, "3_quality": quality_df,
          "4_class_counts": counts_df, "5_paper_vs_ours": paper_vs_ours, "6_duplicates": dup_df,
          "7_split_check": disjoint_df, "8_cross_source_overlap": overlap_df,
          "9_arsarcasm_dialect": dialect_df.reset_index(), "10_arsarcasm_sarcasm": sarcasm_df.reset_index(),
          "11_arsarcasm_source": source_df.reset_index(), "12_saved_files": saved_df}
with pd.ExcelWriter(os.path.join(TABLES, "member1_audit_tables.xlsx")) as xw:
    for sheet, t in tables.items():
        t.to_excel(xw, sheet_name=sheet[:31], index=False)
        t.to_csv(os.path.join(TABLES, sheet + ".csv"), index=False)
conflicts_ars.to_csv(os.path.join(TABLES, "examples_conflicting_duplicates_arsarcasm.csv"), index=False)
label_disagree[["id", "text", "label_arsarcasm", "label_astd"]].to_csv(
    os.path.join(TABLES, "examples_arsarcasm_vs_astd_label_disagreement.csv"), index=False)

def md(t):
    t = t.copy(); t.columns = [str(c) for c in t.columns]
    head = "| " + " | ".join(t.columns) + " |\n|" + "---|" * len(t.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in t.astype(str).values)

report = f"""# Member 1 Data Audit Report
Generated {datetime.datetime.now():%Y-%m-%d %H:%M} · seed {SEED} · dev fraction {DEV_FRAC} · test fraction (AJGT, ASTD) {TEST_FRAC}

## 1. Versions
{md(pd.DataFrame(versions))}

## 2. Label mapping
{md(label_map_df)}

## 3. Basic quality checks
{md(quality_df)}

## 4. Class counts per split
{md(counts_df)}

## 5. Paper sizes vs ours
{md(paper_vs_ours)}

## 6. Duplicates inside each dataset
{md(dup_df)}

ArSarcasm test tweets that also appear in ArSarcasm train (normalised text): {len(ars_leak)}

## 7. Split check
{md(disjoint_df)}

## 8. Cross-source overlap
{md(overlap_df)}

ArSarcasm test tweets also found in ASTD: {len(m)}; labelled differently in the two datasets: {len(label_disagree)}

## 9. Decisions and deviations
- Labels mapped to negative / neutral / positive; ASTD objective tweets removed (following CAMeL Tools, as used by the paper).
- Dev sets: stratified 10% of each training set, seed {SEED}; all copies of the same normalised tweet kept in the same split.
- AJGT and ASTD have no released split; 80/20 stratified train/test with seed {SEED}, so our test sets are not the paper's exact test sets.
- ASTD in the paper is trained on the merged CAMeL-Sent corpus (about 24.8K tweets);
  {'extra sources were provided and merged.' if len(extra_df) else 'ArSAS, SemEval-2017 and ArSenTD-LEV were not available, so ASTD-only training files are provided.'}
- Normalisation is used only to detect duplicates; the saved text is the original text.
"""
with open(os.path.join(OUT, "member1_audit_report.md"), "w", encoding="utf-8") as f:
    f.write(report)
print("Saved: audit report, tables, figures and splits in", OUT)

In [ ]:
# ---------- 14. Zip everything to share with the team (and download it) ----------
zip_path = "member1_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            z.write(os.path.join(root, fn))
print("Created", zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass